# Chapter 11: Context Engineering - Data Strategy at Agent Runtime

This notebook builds a miniature context layer for an AI agent and shows, with
measurable numbers, why *how* you assemble context matters more than how much
you send.

It is fully self-contained: Part 1 creates a small sample project on disk, so it
runs on Google Colab or locally with no downloads and no API key. Only the last
part (comparing real model answers) needs an `OPENAI_API_KEY`, and it skips
itself politely if you do not have one set.

Run the cells top to bottom. Each part matches a section in the chapter.

In [1]:
# === Setup ===
# Standard library only for the core of this chapter. The whole point is that a
# context layer is files and discipline, not another platform.
import json
import re
import shutil
from datetime import date
from pathlib import Path

PROJECT = Path("data/sample_project")

def tokens(text: str) -> int:
    """Rough token estimate (about 4 characters per token for English text).
    Good enough to compare two context assemblies against each other."""
    return len(text) // 4

print("Setup complete. Sample project will live in:", PROJECT)

Setup complete. Sample project will live in: data/sample_project


## Part 1: Create the mini project

We create a small engineering project: two current docs, one meeting note, one
authoritative decision record, and one stale design doc that nobody deleted.
Then we add the three context-layer files from the chapter: `START_HERE.md`,
`Knowledge/KNOWLEDGE_GRAPH.md`, and `AGENTS.md`.

The stale doc is the trap. It describes the authentication design the team
abandoned, and nothing about the file itself says so.

In [2]:
# === Part 1: Create the mini project ===
if PROJECT.exists():
    shutil.rmtree(PROJECT)

FILES = {
    # --- The project's actual documents ---
    "docs/authentication.md": """# Authentication (current)

Every service authenticates through the API gateway using OAuth 2.0
client-credentials. The gateway issues a JWT signed with the platform key,
and services validate the token locally. Tokens expire after 15 minutes.

Do not call the old auth-service directly. It was retired in March 2026
(see Source of Truth/architecture_decisions.md, ADR-014).
""",
    "docs/payments.md": """# Payments

The payments service posts transactions to the ledger through the gateway.
Batch settlement runs at 02:00 UTC. Retries use exponential backoff with a
cap of 5 attempts. Reconciliation reports land in the finance bucket at 06:00.

Payment requests must carry an idempotency key. Duplicate keys within 24 hours
return the original response instead of creating a second charge.
""",
    "meetings/2026-03-02_auth_design_review.md": """# Design review notes, March 2, 2026

Attendees: platform team, security, two service owners.

We reviewed the incident from February where session cookies were replayed
against the legacy auth-service. Security recommended moving every service
to gateway-issued JWTs. Decision recorded as ADR-014. The legacy auth-service
enters read-only mode immediately and is retired at the end of March.
""",
    "Source of Truth/architecture_decisions.md": """# Architecture decision records (authoritative, read-only)

## ADR-014 (March 2026): Gateway-issued JWTs replace the legacy auth-service

All service-to-service authentication goes through the API gateway using
OAuth 2.0 client-credentials with 15-minute JWTs. The legacy auth-service
and its session-cookie flow are retired. Reason: the February replay
incident and the security review that followed.

## ADR-009 (November 2025): One ledger, one writer

Only the payments service writes to the ledger. Every other service reads.
""",
    "archive/old_auth_design.md": """# Authentication design

Services authenticate against the auth-service using session cookies.
The auth-service keeps a session table in Redis with a 24-hour TTL.
On login, the client receives a session cookie, and every downstream
service calls auth-service /validate on each request.

This design is simple and battle-tested. All new services should follow it.
""",
    # --- The context layer (the three files from the chapter) ---
    "START_HERE.md": """# START HERE

This is the payments platform repository. Five services behind one API
gateway, one ledger, batch settlement nightly at 02:00 UTC.

Team: platform (owns the gateway), payments, and reporting.
Credentials: in the team vault, never in this repo.

Agents: read AGENTS.md before doing anything.
""",
    "Knowledge/KNOWLEDGE_GRAPH.md": """# Knowledge Graph

## Document Hierarchy

- Tier 1 (Source of Truth):
  - Source of Truth/architecture_decisions.md
- Tier 2 (Core Knowledge):
  - docs/authentication.md
  - docs/payments.md
- Tier 3 (Working Documents):
  - meetings/2026-03-02_auth_design_review.md
- Tier 4 (Archive, superseded, do not use):
  - archive/old_auth_design.md

## Concept Clusters

- Authentication: Source of Truth/architecture_decisions.md, docs/authentication.md, meetings/2026-03-02_auth_design_review.md
- Payments: Source of Truth/architecture_decisions.md, docs/payments.md

## Evidence Trails

- "We use gateway-issued JWTs" -> meetings/2026-03-02_auth_design_review.md -> Source of Truth/architecture_decisions.md (ADR-014)

## Quick Reference

- "How does authentication work?" -> Authentication cluster
- "How do payments settle?" -> Payments cluster
""",
    "AGENTS.md": """# Rules for AI agents in this workspace

## Session Initialization (MANDATORY)

1. Read START_HERE.md
2. Read Knowledge/KNOWLEDGE_GRAPH.md
3. Read the relevant Source of Truth files
4. Cite sources in file:line format
5. Never speculate. Say "I don't know" when unsure.

Tier 1 overrides every other document. Tier 4 is history, not guidance.
""",
}

for rel, content in FILES.items():
    p = PROJECT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)

total = sum(tokens(c) for c in FILES.values())
print(f"Created {len(FILES)} files under {PROJECT} (about {total:,} tokens of text).")
for rel in FILES:
    print("  ", rel)

Created 8 files under data/sample_project (about 881 tokens of text).
   docs/authentication.md
   docs/payments.md
   meetings/2026-03-02_auth_design_review.md
   Source of Truth/architecture_decisions.md
   archive/old_auth_design.md
   START_HERE.md
   Knowledge/KNOWLEDGE_GRAPH.md
   AGENTS.md


## Part 2: The context dump

The lazy way to give an agent context is to concatenate everything and hope.
This cell does exactly that: it walks the project, stacks every markdown file
into one blob, and reports what the agent would actually see.

Watch two things in the output. The size, and the *order*. The stale design doc
sorts ahead of the current one, and nothing in the blob tells the model which
of the two contradictory authentication designs is real.

In [3]:
# === Part 2: The context dump (naive assembly) ===
def dump_everything(project: Path) -> str:
    parts = []
    for p in sorted(project.rglob("*.md")):
        rel = p.relative_to(project)
        parts.append(f"----- {rel} -----\n{p.read_text()}")
    return "\n".join(parts)

blob = dump_everything(PROJECT)
print(f"Context dump: {tokens(blob):,} tokens from {len(list(PROJECT.rglob('*.md')))} files\n")
print("Order in which the model sees the files:")
for i, p in enumerate(sorted(PROJECT.rglob("*.md")), 1):
    print(f"  {i}. {p.relative_to(PROJECT)}")

# The contradiction the dump hands the model, side by side:
print("\nThe dump contains BOTH of these, with no way to rank them:")
print('  archive/old_auth_design.md : "Services authenticate against the auth-service using session cookies."')
print('  docs/authentication.md     : "OAuth 2.0 client-credentials ... issued by the gateway."')

Context dump: 961 tokens from 8 files

Order in which the model sees the files:
  1. AGENTS.md
  2. Knowledge/KNOWLEDGE_GRAPH.md
  3. START_HERE.md
  4. Source of Truth/architecture_decisions.md
  5. archive/old_auth_design.md
  6. docs/authentication.md
  7. docs/payments.md
  8. meetings/2026-03-02_auth_design_review.md

The dump contains BOTH of these, with no way to rank them:
  archive/old_auth_design.md : "Services authenticate against the auth-service using session cookies."
  docs/authentication.md     : "OAuth 2.0 client-credentials ... issued by the gateway."


## Part 3: Graph-driven assembly

Now the same question goes through the knowledge graph instead. The assembler
reads the graph, finds the concept cluster that matches the question, loads
those files in tier order, and skips Tier 4 entirely. It returns less text and
better text, with citations.

In [4]:
# === Part 3: Graph-driven assembly ===
def parse_graph(project: Path):
    """Parse tiers and concept clusters out of Knowledge/KNOWLEDGE_GRAPH.md."""
    text = (project / "Knowledge/KNOWLEDGE_GRAPH.md").read_text()
    tiers = {}   # file path -> tier number
    tier = None
    in_hierarchy = False
    clusters = {}
    in_clusters = False
    for line in text.splitlines():
        if line.startswith("## "):
            in_hierarchy = "Document Hierarchy" in line
            in_clusters = "Concept Clusters" in line
            continue
        m = re.match(r"- Tier (\d)", line)
        if in_hierarchy and m:
            tier = int(m.group(1))
            continue
        m = re.match(r"\s+- (.+\.md)", line)
        if in_hierarchy and m and tier:
            tiers[m.group(1).strip()] = tier
            continue
        m = re.match(r"- (\w+): (.+)", line)
        if in_clusters and m:
            clusters[m.group(1).lower()] = [f.strip() for f in m.group(2).split(",")]
    return tiers, clusters

def assemble(project: Path, question: str) -> dict:
    tiers, clusters = parse_graph(project)
    topic = next((t for t in clusters if t in question.lower()), None)
    if topic is None:
        return {"error": "No cluster matches. Fall back to search."}
    files = sorted(clusters[topic], key=lambda f: tiers.get(f, 9))  # tier order, archive last
    files = [f for f in files if tiers.get(f, 9) < 4]               # never load Tier 4
    context = "\n".join(
        f"----- {f} (Tier {tiers[f]}) -----\n{(project / f).read_text()}" for f in files
    )
    return {"topic": topic, "files": files, "context": context}

question = "How does authentication work in our system?"
result = assemble(PROJECT, question)

print(f"Question: {question}")
print(f"Matched cluster: {result['topic']}")
print(f"Context: {tokens(result['context']):,} tokens from {len(result['files'])} files (tier order):")
for f in result["files"]:
    print("  ", f)
print(f"\nCompared with the dump: {tokens(blob):,} tokens -> {tokens(result['context']):,} tokens, "
      f"and the superseded design never enters the window.")

Question: How does authentication work in our system?
Matched cluster: authentication
Context: 367 tokens from 3 files (tier order):
   Source of Truth/architecture_decisions.md
   docs/authentication.md
   meetings/2026-03-02_auth_design_review.md

Compared with the dump: 961 tokens -> 367 tokens, and the superseded design never enters the window.


## Part 4: Facts that know when they stopped being true

A knowledge graph can also carry time. This is the temporal drift problem from
Chapter 5, showing up one level higher. Here every fact gets a validity window,
and superseding a fact marks the old one invalid instead of deleting it. This
is a twenty-line version of the idea behind temporal knowledge graph engines
like Graphiti (the open-source engine from the Zep team).

In [5]:
# === Part 4: Facts that know when they stopped being true ===
FACTS = [
    {"fact": "Services authenticate with session cookies via the auth-service",
     "source": "archive/old_auth_design.md",
     "valid_from": date(2024, 6, 1), "invalid_from": None},
]

def supersede(facts, old_substring, new_fact, source, on):
    """Mark the old fact invalid (keep it!) and add the new one."""
    for f in facts:
        if old_substring in f["fact"] and f["invalid_from"] is None:
            f["invalid_from"] = on
    facts.append({"fact": new_fact, "source": source,
                  "valid_from": on, "invalid_from": None})

def facts_as_of(facts, when):
    return [f for f in facts
            if f["valid_from"] <= when and (f["invalid_from"] is None or when < f["invalid_from"])]

# ADR-014 lands in March 2026:
supersede(FACTS, "session cookies",
          "Services authenticate with gateway-issued OAuth 2.0 JWTs (15 min expiry)",
          "Source of Truth/architecture_decisions.md (ADR-014)", on=date(2026, 3, 2))

for when in (date(2026, 1, 15), date(2026, 7, 16)):
    current = facts_as_of(FACTS, when)
    print(f"What is true on {when}:")
    for f in current:
        print(f"  - {f['fact']}  [{f['source']}]")
    print()

print("The old fact is still in the store, marked invalid, so the agent can answer")
print('"why did we move off cookies?" without ever asserting the old design is current.')
invalid = [f for f in FACTS if f["invalid_from"]]
print(f"\nInvalidated facts kept for history: {len(invalid)} "
      f"(invalid since {invalid[0]['invalid_from']})")

What is true on 2026-01-15:
  - Services authenticate with session cookies via the auth-service  [archive/old_auth_design.md]

What is true on 2026-07-16:
  - Services authenticate with gateway-issued OAuth 2.0 JWTs (15 min expiry)  [Source of Truth/architecture_decisions.md (ADR-014)]

The old fact is still in the store, marked invalid, so the agent can answer
"why did we move off cookies?" without ever asserting the old design is current.

Invalidated facts kept for history: 1 (invalid since 2026-03-02)


## Part 5 (optional): Ask a model both ways

If you have an `OPENAI_API_KEY` set, this cell sends the same question twice,
once with the dump and once with the graph-assembled context, and prints both
answers so you can compare them yourself. Without a key, it skips.

In [6]:
# === Part 5 (optional): Ask a model both ways ===
import os

if not os.environ.get("OPENAI_API_KEY"):
    print("No OPENAI_API_KEY set. Skipping the live comparison.")
    print("The measurable part of this chapter (token counts, ordering, tier")
    print("selection, temporal validity) already ran above without any API.")
else:
    from openai import OpenAI
    client = OpenAI()

    def ask(context, question):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system",
                 "content": "Answer ONLY from the provided context. Cite the file you used."},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
            ],
            temperature=0.0,
        )
        return response.choices[0].message.content

    print("=== Answer from the context dump ===")
    print(ask(blob, question))
    print("\n=== Answer from the graph-assembled context ===")
    print(ask(result["context"], question))

No OPENAI_API_KEY set. Skipping the live comparison.
The measurable part of this chapter (token counts, ordering, tier
selection, temporal validity) already ran above without any API.


## Where to go from here

Swap the sample project for one of your own repositories. Start with the three
files (`START_HERE.md`, `Knowledge/KNOWLEDGE_GRAPH.md`, `AGENTS.md`), point
your coding agent at them, and add the temporal validity fields the day a
document gets superseded, not later.

The production version of everything this notebook builds is the open-source
**Agentic Repos** framework: https://github.com/ranyelhousieny/Agentic-Repo

Clone it and run `/project:convert-repo-to-agentic <repo-path-or-url>` (Claude
Code or Windsurf) to generate the full context layer for any repository,
tailored to its tech stack. The chapter walks through how each piece scales up
from this miniature to a production context layer.